In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/Documentation_template.md
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/README.md
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/utils/validate_submission.py
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/dataset/test/test_source2.tsv
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/dataset/test/test_source3.tsv
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/dataset/test/test_source1.tsv
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/dataset/train/train_ground_truth.tsv
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/dataset/train/train_source3.tsv
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/dataset/train/train_source2.tsv
/kaggle/input/datasets/ananyapandey1504/amazondatasetnew/student_resource/data

In [2]:
!pip install -q gdown unidecode lightgbm rapidfuzz xgboost

import os, zipfile, gdown
file_id = "1ChbbSOdZO8g3AtbTF75P0SfvwdjBow40"
out_zip = "/kaggle/working/student_resource.zip"
print("Downloading dataset...")
gdown.download(f"https://drive.google.com/uc?id={file_id}", out_zip, quiet=False)
print("Extracting...")
with zipfile.ZipFile(out_zip) as z: z.extractall("/kaggle/working/")
os.remove(out_zip)
for root, _, files in os.walk("/kaggle/working/"):
    for f in files:
        if f.endswith(".tsv"): print(" ", os.path.join(root, f))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.2 MB/s eta 0:00:00


Downloading...
From (original): https://drive.google.com/uc?id=1ChbbSOdZO8g3AtbTF75P0SfvwdjBow40
From (redirected): https://drive.google.com/uc?id=1ChbbSOdZO8g3AtbTF75P0SfvwdjBow40&confirm=t&uuid=63b8059d-eb42-4bf9-8007-3f589de762f5
To: /kaggle/working/student_resource.zip
100%|██████████| 1.09G/1.09G [00:15<00:00, 72.9MB/s]


Extracting...
  /kaggle/working/student_resource/dataset/train/train_source2.tsv
  /kaggle/working/student_resource/dataset/train/train_source3.tsv
  /kaggle/working/student_resource/dataset/train/train_source1.tsv
  /kaggle/working/student_resource/dataset/train/train_ground_truth.tsv
  /kaggle/working/student_resource/dataset/test/test_source2.tsv
  /kaggle/working/student_resource/dataset/test/test_source3.tsv
  /kaggle/working/student_resource/dataset/test/test_source1.tsv
  /kaggle/working/__MACOSX/student_resource/dataset/train/._train_source1.tsv
  /kaggle/working/__MACOSX/student_resource/dataset/train/._train_source2.tsv
  /kaggle/working/__MACOSX/student_resource/dataset/train/._train_source3.tsv
  /kaggle/working/__MACOSX/student_resource/dataset/train/._train_ground_truth.tsv
  /kaggle/working/__MACOSX/student_resource/dataset/test/._test_source3.tsv
  /kaggle/working/__MACOSX/student_resource/dataset/test/._test_source1.tsv
  /kaggle/working/__MACOSX/student_resource/datas

In [3]:
import os, re, gc, time, math, unicodedata, numpy as np, pandas as pd
from collections import defaultdict, Counter
from unidecode import unidecode
from rapidfuzz import fuzz, distance
from rapidfuzz.process import cpdist
import lightgbm as lgb
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE = "/kaggle/working/student_resource" if os.path.exists("/kaggle/working/student_resource") else "/kaggle/working"
TR, TE = f"{BASE}/dataset/train", f"{BASE}/dataset/test"
OUT = "/kaggle/working/output"; os.makedirs(OUT, exist_ok=True)

SAMPLE_S1  = 50000
NEG_POOL   = 1_500_000    
TOPK_BLOCK = 45          
BLOCK_CAP  = 600
FLOOR      = 0.12         
RS         = 42

LEGAL = sorted(["private limited","public limited","limited liability company","pvt limited","pvt ltd",
                "private ltd","public ltd","incorporated","corporation","company","limited","inc","llc","ltd","corp",
                "plc","llp","gmbh","sarl","sas","pvt","co"], key=len, reverse=True)
LEGAL_PAT = r"\b(" + "|".join(re.escape(x) for x in LEGAL) + r")\b"
DBA_PAT   = r"\b(dba|t\s*/?\s*a|trading as)\b"
GLUE_PAT  = r"(?<=\d)(?=(?:near|opp|opposite|behind|beside)\b)"
ADDR_ABBR = {r"\bst\b":"street", r"\brd\b":"road", r"\bave?\b":"avenue", r"\bblvd\b":"boulevard",
             r"\bdr\b":"drive", r"\bln\b":"lane", r"\bct\b":"court", r"\bhwy\b":"highway",
             r"\bpkwy\b":"parkway", r"\bapt\b":"apartment", r"\bste\b":"suite", r"\bfl\b":"floor", r"\bopp\b":"opposite"}

def _base(s):
    s = s.fillna("").astype(str).map(lambda x: unidecode(unicodedata.normalize("NFKC", x)) if x else "")
    s = s.str.lower().str.replace("&", " and ", regex=False).str.replace("@", " at ", regex=False)
    s = s.str.replace(GLUE_PAT, " ", regex=True).str.replace(r"[^a-z0-9\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def clean_names(s):
    b = _base(s).str.replace(DBA_PAT, " ", regex=True).str.replace(LEGAL_PAT, " ", regex=True)
    return b.str.replace(r"\s+", " ", regex=True).str.strip()

def clean_addrs(s):
    b = _base(s)
    for pat, rep in ADDR_ABBR.items(): b = b.str.replace(pat, rep, regex=True)
    return b.str.replace(r"\s+", " ", regex=True).str.strip()

def _hnum(a):
    m = re.search(r"\d+", a); return m.group(0) if m else ""

def _soundex(name):
    if not name: return ""
    first = name[0].upper()
    mapping = {"b":"1","f":"1","p":"1","v":"1","c":"2","g":"2","j":"2","k":"2","q":"2","s":"2","x":"2","z":"2",
               "d":"3","t":"3","l":"4","m":"5","n":"5","r":"6"}
    code = [first]
    for char in name[1:]:
        s = mapping.get(char, "")
        if s and s != code[-1]: code.append(s)
    return "".join(code[:4]).ljust(4, "0")

def clean_df(df):
    df = df.copy()
    df["name_c"] = clean_names(df["business_name"])
    df["addr_c"] = clean_addrs(df["business_address"])
    df["compact"]= df["name_c"].str.replace(" ", "", regex=False)
    df["country_c"] = df["country"].fillna("").astype(str).str.lower().str.strip()
    df["nums"]   = df["addr_c"].str.findall(r"\d+").apply(set)
    df["postal"] = df["addr_c"].str.findall(r"\b\d{5,6}\b").apply(set)
    df["hnum"]   = df["addr_c"].map(_hnum)
    df["sdx"]    = df["name_c"].map(lambda x: _soundex(x.split()[0]) if x.split() else "")
    return df

def arrays(df):
    return dict(id=df["entity_id"].to_numpy(), name=df["name_c"].to_numpy(),
                addr=df["addr_c"].to_numpy(), compact=df["compact"].to_numpy(),
                country=df["country_c"].to_numpy(), hnum=df["hnum"].to_numpy(),
                sdx=df["sdx"].to_numpy(),
                postal=df["postal"].tolist(), nums=df["nums"].tolist())

def rec_keys(name, compact, nums, postal, addr, sdx):
    ntoks = name.split(); atoks = addr.split(); keys = []
    if compact:
        keys.append(("c", compact))
        if len(compact) >= 5: keys.append(("p", compact[:12]))
    if ntoks:
        keys.append(("s", " ".join(sorted(ntoks))))
        ft = ntoks[0]
        for n in nums:   keys.append(("nt", ft, n))
        for p in postal: keys.append(("pt", ft, p))
    if sdx:
        keys.append(("sdx", sdx))
    for t in ntoks:
        if len(t) >= 4:  keys.append(("t", t))
    for i in range(len(ntoks) - 1):
        keys.append(("b", ntoks[i], ntoks[i + 1]))
    for p in postal:     keys.append(("pa", p))
    if atoks:
        aft = atoks[0]
        for n in nums:   keys.append(("an", aft, n))
        if ntoks:        keys.append(("na", ntoks[0], aft))
    return keys

def build_index(cA):
    idx = defaultdict(list)
    nm, cm, nu, po, ad, sd = cA["name"], cA["compact"], cA["nums"], cA["postal"], cA["addr"], cA["sdx"]
    for i in range(len(nm)):
        for k in rec_keys(nm[i], cm[i], nu[i], po[i], ad[i], sd[i]):
            idx[k].append(i)
    return idx

def block_pairs(names, comps, nums, posts, addrs, sdxs, idx, topk, cap):
    qrep, cpos, votes = [], [], []
    for qi in range(len(names)):
        v = Counter()
        for k in rec_keys(names[qi], comps[qi], nums[qi], posts[qi], addrs[qi], sdxs[qi]):
            lst = idx.get(k)
            if not lst or len(lst) > cap: continue
            for c in lst: v[c] += 1
        if not v: continue
        for c, ct in v.most_common(topk):
            qrep.append(qi); cpos.append(c); votes.append(ct)
    return np.array(qrep, dtype=np.int64), np.array(cpos, dtype=np.int64), np.array(votes, dtype=np.float32)

def build_idf(names):
    df = Counter(); N = max(1, len(names))
    for nm in names:
        for t in set(nm.split()): df[t] += 1
    return {t: math.log(N / (1 + c)) for t, c in df.items()}

FEAT = [
    "vote_count","name_ratio","name_token_sort","name_token_set","name_partial","name_jaro",
    "addr_ratio","addr_token_set","addr_partial",
    "name_tok_jaccard","name_first_eq","name_idf_overlap",
    "addr_tok_jaccard","addr_first_eq","addr_last_eq","house_num_match",
    "number_overlap","postal_match","compact_equal","len_diff",
    "density_gap","density_count","source_flag",
    "name_tfidf_cos", "addr_tfidf_cos", "soundex_eq", "postal_prefix_match",
    "name_len", "addr_len", "name_token_count", "addr_token_count",
    "name_char_jaccard", "addr_char_jaccard", "hnum_len_diff",
    "name_qgram_score", "addr_qgram_score", "name_levenshtein_norm", "addr_levenshtein_norm",
    "num_shared_tokens_name", "num_shared_tokens_addr", "country_match",
    "name_first_token_len", "addr_first_token_len", "name_last_token_len", "addr_last_token_len",
    "digits_ratio_name", "digits_ratio_addr", "compact_len_diff", "name_contain_match"
]

def build_feats(qA, cA, qg, cp, votes, flag, idf, q_tfidf_mat=None, c_tfidf_mat=None, q_atfidf=None, c_atfidf=None):
    n = len(qg)
    qn = qA["name"][qg]; cn = cA["name"][cp]
    qa = qA["addr"][qg]; ca = cA["addr"][cp]
    
    f_nr   = cpdist(qn, cn, scorer=fuzz.ratio,            workers=-1, dtype=np.float32)
    f_nts  = cpdist(qn, cn, scorer=fuzz.token_sort_ratio, workers=-1, dtype=np.float32)
    f_ntse = cpdist(qn, cn, scorer=fuzz.token_set_ratio,  workers=-1, dtype=np.float32)
    f_np   = cpdist(qn, cn, scorer=fuzz.partial_ratio,    workers=-1, dtype=np.float32)
    f_jw   = cpdist(qn, cn, scorer=distance.JaroWinkler.similarity, workers=-1, dtype=np.float32) * 100.0
    f_ar   = cpdist(qa, ca, scorer=fuzz.ratio,            workers=-1, dtype=np.float32)
    f_atse = cpdist(qa, ca, scorer=fuzz.token_set_ratio,  workers=-1, dtype=np.float32)
    f_apar = cpdist(qa, ca, scorer=fuzz.partial_ratio,    workers=-1, dtype=np.float32)
    
    qc = qA["compact"][qg]; cc = cA["compact"][cp]
    f_cmp = np.where((qc != "") & (qc == cc), 100.0, 0.0).astype(np.float32)
    qh = qA["hnum"][qg]; ch = cA["hnum"][cp]
    f_hn  = np.where((qh != "") & (qh == ch), 100.0, 0.0).astype(np.float32)
    qsd = qA["sdx"][qg]; csd = cA["sdx"][cp]
    f_sd  = np.where((qsd != "") & (qsd == csd), 100.0, 0.0).astype(np.float32)
    
    qpost, cpost, qnum, cnum = qA["postal"], cA["postal"], qA["nums"], cA["nums"]
    
    f_njac = np.empty(n, np.float32); f_nfe = np.empty(n, np.float32); f_idf = np.empty(n, np.float32)
    f_ajac = np.empty(n, np.float32); f_afe = np.empty(n, np.float32); f_ale = np.empty(n, np.float32)
    f_post = np.empty(n, np.float32); f_ppost = np.empty(n, np.float32)
    f_num = np.empty(n, np.float32); f_len = np.empty(n, np.float32)
    
    f_n_len = np.empty(n, np.float32); f_a_len = np.empty(n, np.float32)
    f_n_tok_cnt = np.empty(n, np.float32); f_a_tok_cnt = np.empty(n, np.float32)
    f_n_char_jac = np.empty(n, np.float32); f_a_char_jac = np.empty(n, np.float32)
    f_hnum_diff = np.empty(n, np.float32); f_n_qgram = np.empty(n, np.float32); f_a_qgram = np.empty(n, np.float32)
    f_n_lev = np.empty(n, np.float32); f_a_lev = np.empty(n, np.float32)
    f_n_shared = np.empty(n, np.float32); f_a_shared = np.empty(n, np.float32)
    f_country_match = np.empty(n, np.float32)
    f_n_ftok_len = np.empty(n, np.float32); f_a_ftok_len = np.empty(n, np.float32)
    f_n_ltok_len = np.empty(n, np.float32); f_a_ltok_len = np.empty(n, np.float32)
    f_n_dig_ratio = np.empty(n, np.float32); f_a_dig_ratio = np.empty(n, np.float32)
    f_cmp_len = np.empty(n, np.float32); f_contain = np.empty(n, np.float32)

    for i in range(n):
        qnt = qn[i].split(); cnt = cn[i].split()
        sq, sc = set(qnt), set(cnt); inter = sq & sc; uni = sq | sc
        f_njac[i] = len(inter) / (len(uni) if uni else 1) * 100.0
        f_nfe[i]  = 100.0 if (qnt and cnt and qnt[0] == cnt[0]) else 0.0
        f_idf[i]  = sum(idf.get(t, 0.0) for t in inter)
        
        qat = qa[i].split(); cat = ca[i].split()
        saq, sac = set(qat), set(cat); ai = saq & sac; au = saq | sac
        f_ajac[i] = len(ai) / (len(au) if au else 1) * 100.0
        f_afe[i]  = 100.0 if (qat and cat and qat[0] == cat[0]) else 0.0
        f_ale[i]  = 100.0 if (qat and cat and qat[-1] == cat[-1]) else 0.0
        
        gq = int(qg[i]); cci = int(cp[i])
        f_post[i] = 100.0 if (qpost[gq] & cpost[cci]) else 0.0
        
        pfx_match = 0.0
        for p1 in qpost[gq]:
            for p2 in cpost[cci]:
                if len(p1)>=3 and len(p2)>=3 and p1[:3] == p2[:3]: pfx_match = 100.0
        f_ppost[i] = pfx_match

        u = qnum[gq] | cnum[cci]; f_num[i] = len(qnum[gq] & cnum[cci]) / (len(u) if u else 1) * 100.0
        f_len[i]  = len(cn[i]) - len(qn[i])
        
        f_n_len[i] = abs(len(qn[i]) - len(cn[i]))
        f_a_len[i] = abs(len(qa[i]) - len(ca[i]))
        f_n_tok_cnt[i] = abs(len(qnt) - len(cnt))
        f_a_tok_cnt[i] = abs(len(qat) - len(cat))
        
        scq, scc = set(qn[i]), set(cn[i])
        f_n_char_jac[i] = len(scq & scc) / (len(scq | scc) if (scq | scc) else 1) * 100.0
        scaq, scac = set(qa[i]), set(ca[i])
        f_a_char_jac[i] = len(scaq & scac) / (len(scaq | scac) if (scaq | scac) else 1) * 100.0
        
        try:
            f_hnum_diff[i] = abs(int(qh[i]) - int(ch[i])) if (qh[i] and ch[i]) else 0.0
        except:
            f_hnum_diff[i] = 0.0
            
        f_n_qgram[i] = distance.Indel.similarity(qn[i], cn[i]) * 100.0 if (qn[i] and cn[i]) else 0.0
        f_a_qgram[i] = distance.Indel.similarity(qa[i], ca[i]) * 100.0 if (qa[i] and ca[i]) else 0.0
        f_n_lev[i] = distance.Levenshtein.similarity(qn[i], cn[i]) / max(1, max(len(qn[i]), len(cn[i]))) * 100.0
        f_a_lev[i] = distance.Levenshtein.similarity(qa[i], ca[i]) / max(1, max(len(qa[i]), len(ca[i]))) * 100.0
        
        f_n_shared[i] = len(inter)
        f_a_shared[i] = len(ai)
        f_country_match[i] = 100.0 if (qA["country"][gq] == cA["country"][cci]) else 0.0
        
        f_n_ftok_len[i] = abs(len(qnt[0]) if qnt else 0 - len(cnt[0]) if cnt else 0)
        f_a_ftok_len[i] = abs(len(qat[0]) if qat else 0 - len(cat[0]) if cat else 0)
        f_n_ltok_len[i] = abs(len(qnt[-1]) if qnt else 0 - len(cnt[-1]) if cnt else 0)
        f_a_ltok_len[i] = abs(len(qat[-1]) if qat else 0 - len(cat[-1]) if cat else 0)
        
        q_digs = sum(c.isdigit() for c in qn[i]); c_digs = sum(c.isdigit() for c in cn[i])
        f_n_dig_ratio[i] = abs(q_digs - c_digs)
        qa_digs = sum(c.isdigit() for c in qa[i]); ca_digs = sum(c.isdigit() for c in ca[i])
        f_a_dig_ratio[i] = abs(qa_digs - ca_digs)
        
        f_cmp_len[i] = abs(len(qc[i]) - len(cc[i]))
        f_contain[i] = 100.0 if (qn[i] in cn[i] or cn[i] in qn[i]) else 0.0

    if q_tfidf_mat is not None and c_tfidf_mat is not None:
        f_ntfidf = (np.asarray(q_tfidf_mat[qg].multiply(c_tfidf_mat[cp]).sum(1)).ravel() * 100.0).astype(np.float32)
        f_atfidf = (np.asarray(q_atfidf[qg].multiply(c_atfidf[cp]).sum(1)).ravel() * 100.0).astype(np.float32)
    else:
        f_ntfidf = np.zeros(n, dtype=np.float32)
        f_atfidf = np.zeros(n, dtype=np.float32)

    d = pd.DataFrame({"g": qg, "nr": f_nr})
    gmax = d.groupby("g")["nr"].transform("max").to_numpy()
    f_gap = (gmax - f_nr).astype(np.float32)
    d["close"] = (f_nr >= gmax - 5.0).astype(np.float32)
    f_close = d.groupby("g")["close"].transform("sum").to_numpy().astype(np.float32)
    
    return np.column_stack([
        votes, f_nr, f_nts, f_ntse, f_np, f_jw,
        f_ar, f_atse, f_apar,
        f_njac, f_nfe, f_idf,
        f_ajac, f_afe, f_ale, f_hn,
        f_num, f_post, f_cmp, f_len,
        f_gap, f_close, np.full(n, flag, np.float32),
        f_ntfidf, f_atfidf, f_sd, f_ppost,
        f_n_len, f_a_len, f_n_tok_cnt, f_a_tok_cnt,
        f_n_char_jac, f_a_char_jac, f_hnum_diff,
        f_n_qgram, f_a_qgram, f_n_lev, f_a_lev,
        f_n_shared, f_a_shared, f_country_match,
        f_n_ftok_len, f_a_ftok_len, f_n_ltok_len, f_a_ltok_len,
        f_n_dig_ratio, f_a_dig_ratio, f_cmp_len, f_contain
    ]).astype(np.float32)

print("setup ok | base:", BASE, "| features:", len(FEAT))

setup ok | base: /kaggle/working/student_resource | features: 49


In [4]:
t0 = time.time()
s1 = pd.read_csv(f"{TR}/train_source1.tsv", sep="\t")
gt = pd.read_csv(f"{TR}/train_ground_truth.tsv", sep="\t")
gt["matched_entity_ids"] = gt["matched_entity_ids"].fillna("")
GT = {r.source1_entity_id: set(x for x in str(r.matched_entity_ids).split(",") if x) for r in gt.itertuples()}
samp = clean_df(s1.sample(min(SAMPLE_S1, len(s1)), random_state=RS).reset_index(drop=True))
need = set().union(*[GT.get(i, set()) for i in samp["entity_id"]])
needS2 = {x for x in need if x.startswith("S2")}; needS3 = {x for x in need if x.startswith("S3")}
def build_pool(path, need_ids, neg):
    src = pd.read_csv(path, sep="\t")
    must = src[src["entity_id"].isin(need_ids)]
    rest = src[~src["entity_id"].isin(need_ids)].sample(min(neg, len(src)), random_state=RS)
    pool = pd.concat([must, rest]).drop_duplicates("entity_id").reset_index(drop=True)
    del src; gc.collect()
    return clean_df(pool)
poolS2 = build_pool(f"{TR}/train_source2.tsv", needS2, NEG_POOL)
poolS3 = build_pool(f"{TR}/train_source3.tsv", needS3, NEG_POOL)
print(f"sample S1: {len(samp):,} | poolS2: {len(poolS2):,} | poolS3: {len(poolS3):,} | {time.time()-t0:.0f}s")

sample S1: 50,000 | poolS2: 1,583,407 | poolS3: 1,589,229 | 205s


In [5]:
t0 = time.time()
qA = arrays(samp); s2A = arrays(poolS2); s3A = arrays(poolS3)
idf2 = build_idf(s2A["name"]); idf3 = build_idf(s3A["name"])

name_vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
addr_vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))

all_names = np.concatenate([qA["name"], s2A["name"], s3A["name"]])
all_addrs = np.concatenate([qA["addr"], s2A["addr"], s3A["addr"]])
name_vec.fit(all_names)
addr_vec.fit(all_addrs)

q_ntfidf = name_vec.transform(qA["name"])
s2_ntfidf = name_vec.transform(s2A["name"])
s3_ntfidf = name_vec.transform(s3A["name"])

q_atfidf = addr_vec.transform(qA["addr"])
s2_atfidf = addr_vec.transform(s2A["addr"])
s3_atfidf = addr_vec.transform(s3A["addr"])

def block_same_country(qA, cA, topk, cap):
    QR, CP, VT = [], [], []
    for ctry in np.unique(qA["country"]):
        qi = np.where(qA["country"] == ctry)[0]; ci = np.where(cA["country"] == ctry)[0]
        if len(qi) == 0 or len(ci) == 0: continue
        subA = {k: (v[ci] if isinstance(v, np.ndarray) else [v[j] for j in ci]) for k, v in cA.items()}
        idx = build_index(subA)
        qr, cp, vt = block_pairs(qA["name"][qi], qA["compact"][qi], [qA["nums"][j] for j in qi],
                                 [qA["postal"][j] for j in qi], qA["addr"][qi], qA["sdx"][qi], idx, topk, cap)
        if len(qr): QR.append(qi[qr]); CP.append(ci[cp]); VT.append(vt)
        del idx, subA; gc.collect()
    if not QR: return (np.array([],dtype=np.int64),)*2 + (np.array([],np.float32),)
    return np.concatenate(QR), np.concatenate(CP), np.concatenate(VT)

q2, c2, v2 = block_same_country(qA, s2A, TOPK_BLOCK, BLOCK_CAP)
q3, c3, v3 = block_same_country(qA, s3A, TOPK_BLOCK, BLOCK_CAP)
got = defaultdict(set)
for qi, ci in zip(q2, c2): got[int(qi)].add(s2A["id"][int(ci)])
for qi, ci in zip(q3, c3): got[int(qi)].add(s3A["id"][int(ci)])
rec = tot = 0
for k, i in enumerate(qA["id"]):
    truth = GT.get(i, set()) & need
    if not truth: continue
    rec += len(truth & got[k]); tot += len(truth)
print(f"blocking pair-recall (LARGE pool): {rec/max(1,tot)*100:.2f}% | avg cand/business: {(len(q2)+len(q3))/max(1,len(qA['id'])):.1f} | {time.time()-t0:.0f}s")

blocking pair-recall (LARGE pool): 87.17% | avg cand/business: 85.4 | 283s


In [6]:
import time, numpy as np

t0 = time.time()

print("--- Processing S2 Features ---", flush=True)
X2 = build_feats(qA, s2A, q2, c2, v2, 0.0, idf2, q_ntfidf, s2_ntfidf, q_atfidf, s2_atfidf)

print("--- Processing S3 Features ---", flush=True)
X3 = build_feats(qA, s3A, q3, c3, v3, 1.0, idf3, q_ntfidf, s3_ntfidf, q_atfidf, s3_atfidf)

y2 = np.array([1 if s2A["id"][int(c2[i])] in GT.get(qA["id"][int(q2[i])], set()) else 0 for i in range(len(q2))])
y3 = np.array([1 if s3A["id"][int(c3[i])] in GT.get(qA["id"][int(q3[i])], set()) else 0 for i in range(len(q3))])

X = np.vstack([X2, X3])
y = np.concatenate([y2, y3])
grp = np.concatenate([q2, q3]).astype(int)
cand_ids = np.concatenate([s2A["id"][c2], s3A["id"][c3]])

print(f"pairs: {X.shape} | positive rate: {y.mean()*100:.2f}% | total time: {time.time()-t0:.0f}s", flush=True)

--- Processing S2 Features ---
--- Processing S3 Features ---
pairs: (4269233, 49) | positive rate: 3.52% | total time: 204s


In [7]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

print("--- Starting Model Training ---", flush=True)

gkf = GroupKFold(n_splits=5)
oof_lgb = np.zeros(len(y))
oof_xgb = np.zeros(len(y))

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'device': 'gpu',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'seed': RS,
    'verbose': -1
}

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda',
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RS
}

models_lgb = []
models_xgb = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, grp)):
    print(f"--- Training Fold {fold+1}/5 ---", flush=True)
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx], y[val_idx]
    
    # Train LightGBM
    clf_lgb = lgb.train(
        lgb_params,
        lgb.Dataset(X_tr, label=y_tr),
        num_boost_round=1000,
        valid_sets=[lgb.Dataset(X_va, label=y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_lgb[val_idx] = clf_lgb.predict(X_va)
    models_lgb.append(clf_lgb)
    
    # Train XGBoost
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dva = xgb.DMatrix(X_va, label=y_va)
    clf_xgb = xgb.train(
        xgb_params,
        dtr,
        num_boost_round=1000,
        evals=[(dva, "val")],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    oof_xgb[val_idx] = clf_xgb.predict(dva)
    models_xgb.append(clf_xgb)

# Evaluate Validation AUC
auc_lgb = roc_auc_score(y, oof_lgb)
auc_xgb = roc_auc_score(y, oof_xgb)
blend_oof = 0.5 * oof_lgb + 0.5 * oof_xgb
auc_blend = roc_auc_score(y, blend_oof)

print(f"\n[Validation Results]", flush=True)
print(f"LightGBM OOF AUC: {auc_lgb:.5f}", flush=True)
print(f"XGBoost OOF AUC:  {auc_xgb:.5f}", flush=True)
print(f"Blended OOF AUC:  {auc_blend:.5f}", flush=True)

--- Starting Model Training ---
--- Training Fold 1/5 ---


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


--- Training Fold 2/5 ---
--- Training Fold 3/5 ---
--- Training Fold 4/5 ---
--- Training Fold 5/5 ---

[Validation Results]
LightGBM OOF AUC: 0.99983
XGBoost OOF AUC:  0.99987
Blended OOF AUC:  0.99986


In [8]:
from collections import defaultdict
import numpy as np
from sklearn.metrics import precision_score, recall_score

def evaluate_macro_metrics(threshold):
    pred_matches = defaultdict(set)
    for idx, score in enumerate(blend_oof):
        if score >= threshold:
            q_idx = grp[idx]
            q_id = qA["id"][q_idx]
            c_id = cand_ids[idx]
            pred_matches[q_id].add(c_id)
            
    all_q_ids = [qA["id"][i] for i in range(len(qA["id"]))]
    precisions, recalls, f05_scores = [], [], []
    
    for q_id in all_q_ids:
        true_set = GT.get(q_id, set()) & need
        pred_set = pred_matches.get(q_id, set())
        
        # Singleton correct
        if len(true_set) == 0 and len(pred_set) == 0:
            precisions.append(1.0)
            recalls.append(1.0)
            f05_scores.append(1.0)
            continue
        # False positive on singleton or false negative
        if len(true_set) == 0 or len(pred_set) == 0:
            precisions.append(0.0 if len(pred_set) > 0 else 1.0)
            recalls.append(0.0)
            f05_scores.append(0.0)
            continue
            
        tp = len(true_set & pred_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        precisions.append(p)
        recalls.append(r)
        
        if p + r == 0:
            f05_scores.append(0.0)
        else:
            f05_scores.append((1.25 * p * r) / (0.25 * p + r))
            
    return np.mean(precisions), np.mean(recalls), np.mean(f05_scores)

print("--- Evaluating Honest Macro Metrics across Thresholds ---", flush=True)
for th in [0.50, 0.60, 0.70, 0.78, 0.85]:
    p, r, f = evaluate_macro_metrics(th)
    print(f"Threshold {th:.2f} | Macro Precision: {p:.4f} | Macro Recall: {r:.4f} | Macro F0.5: {f:.4f}", flush=True)

--- Evaluating Honest Macro Metrics across Thresholds ---
Threshold 0.50 | Macro Precision: 0.9773 | Macro Recall: 0.8501 | Macro F0.5: 0.9161
Threshold 0.60 | Macro Precision: 0.9826 | Macro Recall: 0.8461 | Macro F0.5: 0.9180
Threshold 0.70 | Macro Precision: 0.9871 | Macro Recall: 0.8405 | Macro F0.5: 0.9184
Threshold 0.78 | Macro Precision: 0.9900 | Macro Recall: 0.8334 | Macro F0.5: 0.9171
Threshold 0.85 | Macro Precision: 0.9930 | Macro Recall: 0.8228 | Macro F0.5: 0.9139


In [9]:
import os, warnings, time, gc
warnings.filterwarnings("ignore")
import pandas as pd, xgboost as xgb
import numpy as np

print("--- Starting GPU-Accelerated Test Inference & Submission Generation ---", flush=True)
OUT = "/kaggle/working/output"
os.makedirs(OUT, exist_ok=True)

OPTIMAL_THRESHOLD = 0.70

uc = ["entity_id","business_name","business_address","country"]
te1 = pd.read_csv(f"{TE}/test_source1.tsv", sep="\t", usecols=uc)
te2 = pd.read_csv(f"{TE}/test_source2.tsv", sep="\t", usecols=uc)
te3 = pd.read_csv(f"{TE}/test_source3.tsv", sep="\t", usecols=uc)
for d in (te1, te2, te3):
    d["country"] = d["country"].fillna("").astype(str)
    d["ckey"] = d["country"].str.lower().str.strip()

def predict_blend_gpu(F):
    # LightGBM prediction
    pl = np.mean([m.predict(F) for m in models_lgb], axis=0)
    
    # XGBoost prediction on GPU (CUDA)
    d_te = xgb.DMatrix(F)
    px = np.mean([m.predict(d_te) for m in models_xgb], axis=0)
    
    return 0.5 * pl + 0.5 * px

# Define the missing best_set function
def best_set(cand_list):
    """Sorts candidates by probability descending and filters out low thresholds or duplicates if needed."""
    if not cand_list:
        return []
    # Sort candidates by probability score descending
    cand_list = sorted(cand_list, key=lambda x: x[0], reverse=True)
    selected = []
    seen_tids = set()
    for p, tid in cand_list:
        if p >= FLOOR and tid not in seen_tids:
            selected.append((p, tid))
            seen_tids.add(tid)
    return selected

CHUNK = 100000
fm = open(f"{OUT}/matching_results.tsv","w"); fm.write("source1_entity_id\tmatched_entity_ids\n")
fc = open(f"{OUT}/candidate_pairs.tsv","w"); fc.write("source1_entity_id\tcandidate_entity_ids\n")
t0 = time.time(); match_counts = []

for ctry in sorted(te1["ckey"].unique()):
    q_df = clean_df(te1[te1["ckey"]==ctry].drop(columns="ckey").reset_index(drop=True))
    qAc = arrays(q_df); nq = len(qAc["id"])
    q_nt = name_vec.transform(qAc["name"]); q_at = addr_vec.transform(qAc["addr"])
    store = [[] for _ in range(nq)]; cand_per_q = [[] for _ in range(nq)]
    
    for raw, flag in [(te2, 0.0), (te3, 1.0)]:
        c_df = clean_df(raw[raw["ckey"]==ctry].drop(columns="ckey").reset_index(drop=True))
        if len(c_df) == 0: del c_df; continue
        cA = arrays(c_df); idx = build_index(cA); idf = build_idf(cA["name"])
        c_nt = name_vec.transform(cA["name"]); c_at = addr_vec.transform(cA["addr"])
        del c_df; gc.collect()
        
        for s in range(0, nq, CHUNK):
            e = min(s+CHUNK, nq)
            qr, cp, vt = block_pairs(qAc["name"][s:e], qAc["compact"][s:e],
                [qAc["nums"][j] for j in range(s,e)], [qAc["postal"][j] for j in range(s,e)],
                qAc["addr"][s:e], qAc["sdx"][s:e], idx, TOPK_BLOCK, BLOCK_CAP)
            if len(qr):
                qg = qr + s
                F = build_feats(qAc, cA, qg, cp, vt, flag, idf, q_nt, c_nt, q_at, c_at)
                pr = predict_blend_gpu(F)
                cids = cA["id"][cp]
                for i in range(len(qr)):
                    gloc = s + int(qr[i]); cand_per_q[gloc].append(cids[i])
                    if pr[i] >= FLOOR: store[gloc].append((float(pr[i]), cids[i]))
            print(f"  [{ctry}] src{int(flag)} chunk {s//CHUNK+1} | {time.time()-t0:.0f}s", flush=True)
        del cA, idx, idf, c_nt, c_at; gc.collect()

    chosen = []
    for loc in range(nq):
        for p, tid in best_set(store[loc]):
            chosen.append((p, loc, tid))
    chosen.sort(key=lambda z: z[0], reverse=True)
    used = set(); assigned = [[] for _ in range(nq)]
    for p, loc, tid in chosen:
        if tid in used: continue
        used.add(tid); assigned[loc].append(tid)
        
    for loc in range(nq):
        final_matches = [tid for p, tid in store[loc] if p >= OPTIMAL_THRESHOLD and tid in assigned[loc]]
        fm.write(f"{qAc['id'][loc]}\t{','.join(final_matches)}\n")
        fc.write(f"{qAc['id'][loc]}\t{','.join(cand_per_q[loc])}\n")
        match_counts.append(len(final_matches))
    fm.flush(); fc.flush()
    print(f"=== {ctry} DONE | {time.time()-t0:.0f}s ===", flush=True)
    del qAc, q_df, q_nt, q_at, store, cand_per_q, chosen, assigned; gc.collect()

fm.close(); fc.close()
mc = np.array(match_counts)
print(f"\n[Success!] GPU Test Inference Complete! {time.time()-t0:.0f}s | singleton ratio: {(mc==0).mean():.3f} | avg matches: {mc.mean():.2f}", flush=True)
print(f"Generated files saved in: {OUT}", flush=True)

--- Starting GPU-Accelerated Test Inference & Submission Generation ---
  [france] src0 chunk 1 | 355s
  [france] src0 chunk 2 | 632s
  [france] src0 chunk 3 | 797s
  [france] src1 chunk 1 | 1142s
  [france] src1 chunk 2 | 1418s
  [france] src1 chunk 3 | 1581s
=== france DONE | 1597s ===
  [india] src0 chunk 1 | 2261s
  [india] src0 chunk 2 | 2594s
  [india] src0 chunk 3 | 2928s
  [india] src0 chunk 4 | 3260s
  [india] src0 chunk 5 | 3591s
  [india] src0 chunk 6 | 3920s
  [india] src0 chunk 7 | 4250s
  [india] src0 chunk 8 | 4578s
  [india] src0 chunk 9 | 4610s
  [india] src1 chunk 1 | 5183s
  [india] src1 chunk 2 | 5504s
  [india] src1 chunk 3 | 5824s
  [india] src1 chunk 4 | 6141s
  [india] src1 chunk 5 | 6463s
  [india] src1 chunk 6 | 6782s
  [india] src1 chunk 7 | 7108s
  [india] src1 chunk 8 | 7434s
  [india] src1 chunk 9 | 7466s
=== india DONE | 7491s ===
  [us] src0 chunk 1 | 7952s
  [us] src0 chunk 2 | 8224s
  [us] src0 chunk 3 | 8498s
  [us] src0 chunk 4 | 8771s
  [us] src0 ch